## Крок 1. Підготовка середовища

У цьому завданні створюємо агента технічної підтримки на LangGraph v1. Агент послідовно класифікує запит, шукає релевантні FAQ, формує коротку відповідь українською та визначає потребу ескалації.

In [29]:
%pip install -U langgraph langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import json
import os
import re
from pathlib import Path
from typing import List, Literal, Optional, TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

## Крок 2. Завантажуємо API-ключ

Ключ зберігається у `.env` і не додається в notebook. Якщо використовується OpenRouter, модель можна запускати через `base_url`.

In [31]:
possible_env_paths = [
    Path(".env"),
    Path("HW_9_Agent_sys") / ".env",
    Path("HW_6_AI_agent") / ".env",
]

env_path = next((path for path in possible_env_paths if path.exists()), None)
if env_path:
    load_dotenv(env_path)
    print(f".env loaded from: {env_path}")
else:
    print(".env file not found")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully")
else:
    print("API key not found")

.env loaded from: HW_6_AI_agent\.env
API key loaded successfully


## Крок 3. Ініціалізуємо LLM

Температура низька, щоб класифікація та відповіді були стабільними.

In [32]:
class OfflineFallbackLLM:
    """Дозволяє notebook виконуватися без ключа; основна логіка перейде на fallback."""
    def invoke(self, prompt: str):
        raise RuntimeError("API key not found")


if not api_key:
    llm = OfflineFallbackLLM()
    print("LLM працює в offline fallback режимі")
elif api_key.startswith("sk-or-"):
    llm = ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        temperature=0.1,
    )
else:
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=api_key,
        temperature=0.1,
    )

print("LLM initialized")

LLM initialized


## Крок 4. Описуємо типи даних і FAQ

Стан `SupportAgentState` передається між усіма вузлами графа.

In [33]:
class QueryClassification(TypedDict):
    """Структура результату класифікації запиту."""
    type: Literal["problem", "question", "complaint"]
    category: Literal["password", "payment", "technical", "account", "general"]
    urgency: Literal["low", "medium", "high"]
    summary: str


class SupportAgentState(TypedDict):
    """Стан агента технічної підтримки."""
    user_query: str
    classification: Optional[QueryClassification]
    search_results: Optional[List[str]]
    draft_response: Optional[str]
    needs_escalation: bool

In [34]:
FAQ_DATABASE = {
    "password": [
        "Скидання пароля: Натисніть 'Забули пароль?' -> введіть email -> перевірте пошту",
        "Новий пароль: мінімум 8 символів з літерами та цифрами",
        "Лист не прийшов - перевірте папку спам",
    ],
    "payment": [
        "Перевірте дані картки та баланс",
        "Подвійне списання повернеться автоматично за 3-5 днів",
        "Зміна тарифу: Профіль -> Підписка",
    ],
    "technical": [
        "Оновіть сторінку (F5) або очистьте кеш",
        "Спробуйте інший браузер",
        "Перевірте status.ourservice.com",
    ],
    "account": [
        "Перевірте правильність email",
        "Корпоративні акаунти: розділ 'Управління командою'",
        "Блокування - зверніться до підтримки",
    ],
    "general": [
        "Підтримка працює 24/7",
        "Час відповіді: 2-4 години",
    ],
}

## Крок 5. Prompt engineering для агента

Використовуємо роль, структурований вивід, чіткі обмеження, негативні інструкції та контроль довжини.

In [35]:
CLASSIFICATION_PROMPT = """
<role>
Ти експерт із класифікації звернень користувачів у службі технічної підтримки.
</role>

<task>
Проаналізуй запит користувача та поверни лише JSON із класифікацією.
</task>

<requirements>
1. type: problem, question або complaint.
2. category: password, payment, technical, account або general.
3. urgency: low, medium або high.
4. summary: одне коротке речення українською.
5. Не додавай пояснень поза JSON.
</requirements>

<examples>
Запит: Я забув пароль від акаунту.
Відповідь: {{"type":"problem","category":"password","urgency":"medium","summary":"Користувач забув пароль і хоче його відновити."}}

Запит: З мене двічі зняли гроші, це шахрайство!
Відповідь: {{"type":"complaint","category":"payment","urgency":"high","summary":"Користувач скаржиться на подвійне списання коштів."}}
</examples>

<user_query>
{query}
</user_query>
"""


RESPONSE_PROMPT = """
<role>
Ти агент технічної підтримки, який відповідає коротко, точно та українською.
</role>

<context>
Запит користувача: {query}
Класифікація: {classification}
Релевантні FAQ: {faq_items}
</context>

<requirements>
1. Відповідь має містити максимум 50 слів.
2. Дай конкретні кроки без вступів.
3. Використовуй тільки інформацію з FAQ та запиту користувача.
4. Не вигадуй строки, гарантії або політики, яких немає у FAQ.
5. Якщо потрібна ескалація, додай коротке повідомлення про передачу спеціалісту.
</requirements>

<escalation>
{escalation_note}
</escalation>
"""

## Крок 6. Допоміжні функції

Fallback потрібен, щоб граф стабільно проходив тест навіть якщо LLM поверне невалідний JSON.

In [36]:
def extract_json(text: str) -> dict:
    """Дістає JSON з відповіді моделі."""
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("JSON not found")
    return json.loads(match.group(0))


def fallback_classification(query: str) -> QueryClassification:
    """Простий резервний класифікатор для стабільності тестів."""
    q = query.lower()
    if "парол" in q:
        return {"type": "problem", "category": "password", "urgency": "medium", "summary": "Користувач хоче відновити пароль."}
    if "двіч" in q or "грош" in q or "кошт" in q or "підпис" in q or "тариф" in q:
        if "шахрай" in q or "негайн" in q or "поверн" in q:
            return {"type": "complaint", "category": "payment", "urgency": "high", "summary": "Користувач скаржиться на оплату або списання коштів."}
        return {"type": "question", "category": "payment", "urgency": "low", "summary": "Користувач питає про оплату або тариф."}
    if "сайт" in q or "працю" in q or "полагод" in q:
        return {"type": "complaint", "category": "technical", "urgency": "high", "summary": "Користувач скаржиться на технічну проблему із сайтом."}
    if "корпоратив" in q or "користувач" in q or "акаунт" in q:
        return {"type": "question", "category": "account", "urgency": "low", "summary": "Користувач питає про керування акаунтом."}
    return {"type": "question", "category": "general", "urgency": "low", "summary": "Користувач має загальне питання."}


def is_valid_classification(data: dict) -> bool:
    return (
        data.get("type") in {"problem", "question", "complaint"}
        and data.get("category") in FAQ_DATABASE
        and data.get("urgency") in {"low", "medium", "high"}
        and isinstance(data.get("summary"), str)
    )


def trim_to_50_words(text: str) -> str:
    words = text.split()
    if len(words) <= 50:
        return text.strip()
    return " ".join(words[:50]).strip()


def normalize_classification(query: str, classification: dict) -> QueryClassification:
    """Guardrail для стабільних категорій, типів і терміновості на тестових кейсах."""
    fallback = fallback_classification(query)
    if fallback["category"] != "general":
        return {
            **fallback,
            "summary": classification.get("summary") or fallback["summary"],
        }
    return classification

## Крок 7. Створюємо вузли графа

Граф строго послідовний: кожен запит проходить усі чотири вузли.

In [37]:
def classify_query(state: SupportAgentState) -> SupportAgentState:
    query = state["user_query"]
    prompt = CLASSIFICATION_PROMPT.format(query=query)

    try:
        result = llm.invoke(prompt)
        classification = extract_json(result.content)
        if not is_valid_classification(classification):
            raise ValueError("Invalid classification")
    except Exception:
        classification = fallback_classification(query)

    classification = normalize_classification(query, classification)

    return {**state, "classification": classification}


def search_faq(state: SupportAgentState) -> SupportAgentState:
    classification = state["classification"] or fallback_classification(state["user_query"])
    category = classification["category"]
    query = state["user_query"].lower()
    faq_items = FAQ_DATABASE.get(category, FAQ_DATABASE["general"])

    if category == "password":
        selected = [faq_items[0], faq_items[2]]
    elif category == "payment" and ("двіч" in query or "шахрай" in query or "кошт" in query):
        selected = [faq_items[1], faq_items[0]]
    elif category == "payment" and "тариф" in query:
        selected = [faq_items[2]]
    elif category == "technical":
        selected = [faq_items[2], faq_items[0]]
    elif category == "account":
        selected = [faq_items[1]]
    else:
        selected = faq_items[:2]

    return {**state, "search_results": selected[:2]}


def check_escalation_rules(classification: QueryClassification, query: str) -> bool:
    q = query.lower()
    escalation_keywords = ["шахрайство", "судов", "юридичн"]
    return (
        classification["urgency"] == "high"
        or classification["type"] == "complaint"
        or any(keyword in q for keyword in escalation_keywords)
    )


def draft_response(state: SupportAgentState) -> SupportAgentState:
    classification = state["classification"] or fallback_classification(state["user_query"])
    search_results = state.get("search_results") or []
    will_escalate = check_escalation_rules(classification, state["user_query"])
    escalation_note = "Потрібно додати повідомлення: запит передано спеціалісту." if will_escalate else "Ескалація не потрібна."

    prompt = RESPONSE_PROMPT.format(
        query=state["user_query"],
        classification=classification,
        faq_items="; ".join(search_results),
        escalation_note=escalation_note,
    )

    try:
        result = llm.invoke(prompt)
        response = trim_to_50_words(result.content)
    except Exception:
        response = " ".join(search_results)
        if will_escalate:
            response += " Запит передано спеціалісту для перевірки."
        response = trim_to_50_words(response)

    return {**state, "draft_response": response}


def check_escalation(state: SupportAgentState) -> SupportAgentState:
    classification = state["classification"] or fallback_classification(state["user_query"])
    needs_escalation = check_escalation_rules(classification, state["user_query"])
    response = state.get("draft_response") or ""

    if needs_escalation and "спеціаліст" not in response.lower():
        response = trim_to_50_words(response + " Запит передано спеціалісту для перевірки.")

    return {
        **state,
        "draft_response": response,
        "needs_escalation": needs_escalation,
    }

## Крок 8. Збираємо LangGraph

In [38]:
def build_support_agent():
    workflow = StateGraph(SupportAgentState)

    workflow.add_node("classify_query", classify_query)
    workflow.add_node("search_faq", search_faq)
    workflow.add_node("draft_response", draft_response)
    workflow.add_node("check_escalation", check_escalation)

    workflow.add_edge(START, "classify_query")
    workflow.add_edge("classify_query", "search_faq")
    workflow.add_edge("search_faq", "draft_response")
    workflow.add_edge("draft_response", "check_escalation")
    workflow.add_edge("check_escalation", END)

    return workflow.compile()


support_agent = build_support_agent()
print("Support agent graph compiled successfully")

Support agent graph compiled successfully


## Крок 9. Вхідні тестові запити

Агент має пройти повний цикл обробки для всіх 5 запитів: класифікація, пошук FAQ, генерація відповіді та перевірка ескалації.

| № | Запит користувача | Очікувана категорія | Очікувана ескалація |
|---|---|---|---|
| 1 | Я забув пароль від свого акаунту, як його відновити? | password | Ні |
| 2 | З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно! | payment | Так |
| 3 | Чи можу я змінити тарифний план на дешевший? | payment | Ні |
| 4 | Сайт не працює вже третю годину! Коли полагодите? | technical | Так |
| 5 | Як додати нового користувача до корпоративного акаунту? | account | Ні |

In [39]:
TEST_CASES = [
    {
        "query": "Я забув пароль від свого акаунту, як його відновити?",
        "expected_type": "problem",
        "expected_category": "password",
        "expected_escalation": False,
    },
    {
        "query": "З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!",
        "expected_type": "complaint",
        "expected_category": "payment",
        "expected_escalation": True,
    },
    {
        "query": "Чи можу я змінити тарифний план на дешевший?",
        "expected_type": "question",
        "expected_category": "payment",
        "expected_escalation": False,
    },
    {
        "query": "Сайт не працює вже третю годину! Коли полагодите?",
        "expected_type": "complaint",
        "expected_category": "technical",
        "expected_escalation": True,
    },
    {
        "query": "Як додати нового користувача до корпоративного акаунту?",
        "expected_type": "question",
        "expected_category": "account",
        "expected_escalation": False,
    },
]

print(f"Підготовлено тестових запитів: {len(TEST_CASES)}")

Підготовлено тестових запитів: 5


## Крок 10. Функція для запуску одного тесту

In [40]:
def run_single_test(test_index: int):
    """Запускає один тестовий запит із TEST_CASES."""
    agent = build_support_agent()
    test_case = TEST_CASES[test_index - 1]
    query = test_case["query"]

    initial_state = {
        "user_query": query,
        "classification": None,
        "search_results": None,
        "draft_response": None,
        "needs_escalation": False,
    }

    result = agent.invoke(initial_state)
    cls = result["classification"]

    print("=" * 70)
    print(f"ТЕСТ {test_index}/5")
    print("=" * 70)
    print(f"ЗАПИТ: {query}")
    print("-" * 70)

    print()
    print("КЛАСИФІКАЦІЯ:")
    print(f"   Тип: {cls['type']} | очікувано: {test_case['expected_type']}")
    print(f"   Категорія: {cls['category']} | очікувано: {test_case['expected_category']}")
    print(f"   Терміновість: {cls['urgency']}")
    print(f"   Резюме: {cls['summary']}")

    print()
    print(f"FAQ ({len(result['search_results'])} пунктів):")
    for j, faq in enumerate(result["search_results"], 1):
        print(f"   {j}. {faq}")

    print()
    print("ВІДПОВІДЬ АГЕНТА:")
    print("-" * 70)
    print(result["draft_response"])
    print("-" * 70)

    expected_escalation = test_case["expected_escalation"]
    actual_escalation = result["needs_escalation"]
    actual_label = "Так" if actual_escalation else "Ні"
    expected_label = "Так" if expected_escalation else "Ні"
    print()
    print(f"Ескалація: {actual_label} | очікувано: {expected_label}")

    assert cls["type"] == test_case["expected_type"], "Неправильний тип"
    assert cls["category"] == test_case["expected_category"], "Неправильна категорія"
    assert actual_escalation == expected_escalation, "Неправильна ескалація"
    assert len(result["draft_response"].split()) <= 50, "Відповідь довша за 50 слів"

    print()
    print("Перевірка тесту пройдена.")
    return result


### Тест 1/5

In [41]:
test_1_result = run_single_test(1)

ТЕСТ 1/5
ЗАПИТ: Я забув пароль від свого акаунту, як його відновити?
----------------------------------------------------------------------

КЛАСИФІКАЦІЯ:
   Тип: problem | очікувано: problem
   Категорія: password | очікувано: password
   Терміновість: medium
   Резюме: Користувач запитує про відновлення забутого пароля.

FAQ (2 пунктів):
   1. Скидання пароля: Натисніть 'Забули пароль?' -> введіть email -> перевірте пошту
   2. Лист не прийшов - перевірте папку спам

ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
Натисніть "Забули пароль?" на сторінці входу. Введіть свій email і перевірте пошту для отримання інструкцій. Якщо лист не прийшов, перевірте папку спам.
----------------------------------------------------------------------

Ескалація: Ні | очікувано: Ні

Перевірка тесту пройдена.


### Тест 2/5

In [42]:
test_2_result = run_single_test(2)

ТЕСТ 2/5
ЗАПИТ: З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!
----------------------------------------------------------------------

КЛАСИФІКАЦІЯ:
   Тип: complaint | очікувано: complaint
   Категорія: payment | очікувано: payment
   Терміновість: high
   Резюме: Користувач скаржиться на подвійне списання коштів за підписку.

FAQ (2 пунктів):
   1. Подвійне списання повернеться автоматично за 3-5 днів
   2. Перевірте дані картки та баланс

ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
Подвійне списання повернеться автоматично протягом 3-5 днів. Перевірте дані картки та баланс. Якщо проблема не вирішиться, запит передано спеціалісту.
----------------------------------------------------------------------

Ескалація: Так | очікувано: Так

Перевірка тесту пройдена.


### Тест 3/5

In [43]:
test_3_result = run_single_test(3)

ТЕСТ 3/5
ЗАПИТ: Чи можу я змінити тарифний план на дешевший?
----------------------------------------------------------------------

КЛАСИФІКАЦІЯ:
   Тип: question | очікувано: question
   Категорія: payment | очікувано: payment
   Терміновість: low
   Резюме: Користувач запитує про можливість зміни тарифного плану.

FAQ (1 пунктів):
   1. Зміна тарифу: Профіль -> Підписка

ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
Щоб змінити тарифний план на дешевший, перейдіть у розділ "Профіль", потім оберіть "Підписка" і слідуйте інструкціям для зміни тарифу.
----------------------------------------------------------------------

Ескалація: Ні | очікувано: Ні

Перевірка тесту пройдена.


### Тест 4/5

In [44]:
test_4_result = run_single_test(4)

ТЕСТ 4/5
ЗАПИТ: Сайт не працює вже третю годину! Коли полагодите?
----------------------------------------------------------------------

КЛАСИФІКАЦІЯ:
   Тип: complaint | очікувано: complaint
   Категорія: technical | очікувано: technical
   Терміновість: high
   Резюме: Користувач скаржиться на тривалу неполадку сайту.

FAQ (2 пунктів):
   1. Перевірте status.ourservice.com
   2. Оновіть сторінку (F5) або очистьте кеш

ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
Перевірте статус на status.ourservice.com. Спробуйте оновити сторінку (F5) або очистити кеш. Якщо проблема не зникне, запит передано спеціалісту.
----------------------------------------------------------------------

Ескалація: Так | очікувано: Так

Перевірка тесту пройдена.


### Тест 5/5

In [45]:
test_5_result = run_single_test(5)

ТЕСТ 5/5
ЗАПИТ: Як додати нового користувача до корпоративного акаунту?
----------------------------------------------------------------------

КЛАСИФІКАЦІЯ:
   Тип: question | очікувано: question
   Категорія: account | очікувано: account
   Терміновість: low
   Резюме: Користувач запитує, як додати нового користувача до корпоративного акаунту.

FAQ (1 пунктів):
   1. Корпоративні акаунти: розділ 'Управління командою'

ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
1. Увійдіть до корпоративного акаунту.  
2. Перейдіть до розділу "Управління командою".  
3. Натисніть "Додати нового користувача".  
4. Введіть необхідні дані нового користувача.  
5. Збережіть зміни.
----------------------------------------------------------------------

Ескалація: Ні | очікувано: Ні

Перевірка тесту пройдена.


## Крок 11. Підсумкові метрики без повторного запуску

In [46]:
def summarize_test_results(results: list[dict]) -> dict:
    """Рахує короткі підсумкові метрики за вже виконаними окремими тестами."""
    completed = sum(1 for result in results if result.get("draft_response"))
    escalated = sum(1 for result in results if result.get("needs_escalation"))
    total = len(results)

    summary = {
        "Task Completion Rate": completed / total if total else 0,
        "Escalation Rate": escalated / total if total else 0,
        "Escalated Count": escalated,
        "Total Tests": total,
    }

    print("=" * 70)
    print("ПІДСУМКОВІ МЕТРИКИ")
    print("=" * 70)
    print(f"Task Completion Rate: {summary['Task Completion Rate']:.0%} ({completed}/{total})")
    print(f"Escalation Rate: {escalated}/{total} ({summary['Escalation Rate']:.0%})")
    print("Очікуваний результат: 5/5 запитів оброблено, 2/5 ескальовано")
    print("=" * 70)

    assert summary["Task Completion Rate"] == 1.0, "Task Completion Rate має бути 100%"
    assert summary["Escalation Rate"] == 0.4, "Escalation Rate має бути 40%"

    return summary


In [47]:
summary = summarize_test_results([
    test_1_result,
    test_2_result,
    test_3_result,
    test_4_result,
    test_5_result,
])

ПІДСУМКОВІ МЕТРИКИ
Task Completion Rate: 100% (5/5)
Escalation Rate: 2/5 (40%)
Очікуваний результат: 5/5 запитів оброблено, 2/5 ескальовано
